In [2]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1）第1正規化が必要な例

学生情報と試験の点数を、一つのテーブルに保存すると仮定する。

#### [方法1] 一つの列に複数の点数を保存する

| 学籍番号 | 氏名 | 点数 |
|---|---|---|
| S001 | 千尋 | 90, 88, 95 |
| S002 | ハウル | 85, 92 |
| S003 | ソフィー | 78, 81, 88, 91 |

#### ! 問題点

- 一つの列に複数の値が保存されている。
- 特定の試験の点数だけを検索しにくい。
- 平均点や最高点を計算しにくい。
- 値の追加・修正には文字列処理が必要となり、SQLが複雑になる。

> -> データが増えるほど、検索・更新・削除・集計が難しくなる。

## [方法2] 点数の列を複数作成する

| 学籍番号 | 氏名 | 点数1 | 点数2 | 点数3 | 点数4 |
|---|---|---:|---:|---:|---:|
| S001 | 千尋 | 90 | 88 | 95 | NULL |
| S002 | ハウル | 85 | 92 | NULL | NULL |
| S003 | ソフィー | 78 | 81 | 88 | 91 |

#### ! 問題点

- 試験回数が増えるたびに列を追加する必要があり、拡張性が低い。
- 学生ごとに試験回数が異なるため、`NULL`が多く発生する。
- 「3回目の試験」のように、列の位置に意味が依存する。
- 試験別の検索や集計が難しい。

> -> データが増えるほど、検索・更新・削除・集計が難しくなる。

## [適切な設計]

学生情報と試験情報を、それぞれ別のテーブルに分離して保存する。

#### # 学生テーブル

| 学籍番号 | 氏名 |
|---|---|
| S001 | 千尋 |
| S002 | ハウル |
| S003 | ソフィー |

### # 試験テーブル

| 学籍番号 | 試験番号 | 点数 |
|---|---:|---:|
| S001 | 1 | 90 |
| S001 | 2 | 88 |
| S001 | 3 | 95 |
| S002 | 1 | 85 |
| S002 | 2 | 92 |
| S003 | 1 | 78 |
| S003 | 2 | 81 |
| S003 | 3 | 88 |
| S003 | 4 | 91 |

#### 長所
- 試験が増えても、列ではなく行を追加するだけでよい。
- 検索・集計が容易になり、学生情報と試験情報を分けて管理できる。
- データの重複や管理コストを減らせる。

## 2. 第1正規化の実習

#### 1）原子的でない列を1:Nの関係に分離する

- 一つの列には、一つの値である`原子値`だけを保存しなければならない。

- 一つの列に複数の値を保存すると、検索・更新・削除が難しくなる。


In [10]:
%%sql

CREATE TABLE person (
    person_id INT PRIMARY KEY,
    person_name VARCHAR(30),
    hobby VARCHAR(100)
);

++
||
++
++

In [11]:
%%sql

INSERT INTO person VALUES
(1001, '千尋', '野球,サッカー'),
(1002, 'ハウル', '読書'),
(1003, 'ソフィー', '映画鑑賞,京都旅行'),
(1004, 'パズー', '登山,水泳,北海道キャンプ'),
(1005, 'キキ', 'ゲーム,草野球');

++
||
++
++

#### [問題点]

- 特定の趣味を検索することが難しい。
- 修正や削除も難しい。

In [12]:
%%sql

SELECT *
FROM test.person
WHERE hobby = '野球';  -- 検索結果がない

person_id,person_name,hobby


In [13]:
%%sql

SELECT *
FROM test.person
WHERE hobby LIKE '%野球%';  -- 草野球も検索結果に含まれる

person_id,person_name,hobby
1001,千尋,"野球,サッカー"
1005,キキ,"ゲーム,草野球"


## `第1正規形に分離`

In [14]:
%%sql
CREATE TABLE person1 (
    person_id INT PRIMARY KEY,
    person_name VARCHAR(30)
);

++
||
++
++

In [15]:
%%sql

INSERT INTO person1 VALUES
(1001, '千尋'),
(1002, 'ハウル'),
(1003, 'ソフィー'),
(1004, 'パズー'),
(1005, 'キキ');

++
||
++
++

In [16]:
%%sql

CREATE TABLE person1_hobby (
    person_id INT,
    hobby VARCHAR(30),
    PRIMARY KEY (person_id, hobby),
    FOREIGN KEY (person_id)
        REFERENCES test.person1(person_id)
);

++
||
++
++

In [17]:
%%sql

INSERT INTO person1_hobby VALUES
(1001, '野球'),
(1001, 'サッカー'),
(1002, '読書'),
(1003, '映画鑑賞'),
(1003, '旅行'),
(1004, '登山'),
(1004, '水泳'),
(1004, 'キャンプ'),
(1005, 'ゲーム'),
(1005, '草野球');

++
||
++
++

In [ ]:
%%sql

SELECT *
FROM person1 p
JOIN person1_hobby ph
    ON p.person_id = ph.person_id;

person_id,person_name,person_id_1,hobby
1001,千尋,1001,サッカー
1001,千尋,1001,野球
1002,ハウル,1002,読書
1003,ソフィー,1003,旅行
1003,ソフィー,1003,映画鑑賞
1004,パズー,1004,キャンプ
1004,パズー,1004,水泳
1004,パズー,1004,登山
1005,キキ,1005,ゲーム
1005,キキ,1005,草野球


In [18]:
%%sql

-- 取得できる
SELECT *
FROM person1_hobby
WHERE hobby = '映画鑑賞';

person_id,hobby
1003,映画鑑賞


In [27]:
%%sql

-- 人物の詳細情報も確認する場合はJOIN
SELECT *
FROM person1 p
JOIN person1_hobby ph
    ON p.person_id = ph.person_id
WHERE hobby = '映画鑑賞';

person_id,person_name,person_id_1,hobby
1003,ソフィー,1003,映画鑑賞


## 2）繰り返し列を1:Nの関係に分離する

学生が最大3科目まで履修すると仮定する。

In [20]:
%%sql

CREATE TABLE test.student (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(30),
    subject1 VARCHAR(30),
    subject2 VARCHAR(30),
    subject3 VARCHAR(30)
);

++
||
++
++

In [21]:
%%sql

INSERT INTO test.student VALUES
(1001, '千尋', 'DB', 'Python', 'Power BI'),
(1002, 'ハウル', 'Java', 'SQL', NULL),
(1003, 'キキ', 'Python', NULL, NULL);

++
||
++
++

In [22]:
%%sql

SELECT *
FROM test.student;

student_id,student_name,subject1,subject2,subject3
1001,千尋,DB,Python,Power BI
1002,ハウル,Java,SQL,None
1003,キキ,Python,None,None


#### ! 問題点

- 科目が増えるたびに列を追加する必要がある。
- 履修科目が少ない学生には`NULL`が発生する。
- 保存できる科目数が制限される。
- 検索や集計のSQLが複雑になる。

## `第1正規形に分離`

In [28]:
%%sql

CREATE TABLE test.student1 (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(30)
);

++
||
++
++

In [30]:
%%sql

INSERT INTO test.student1 VALUES
(1001, '千尋'),
(1002, 'ハウル'),
(1003, 'キキ');

++
||
++
++

In [31]:
%%sql

CREATE TABLE test.student1_subject (
    student_id INT,
    subject_name VARCHAR(30),
    PRIMARY KEY (student_id, subject_name),
    FOREIGN KEY (student_id)
        REFERENCES test.student1(student_id)
);

++
||
++
++

In [32]:
%%sql

INSERT INTO test.student1_subject VALUES
(1001, 'DB'),
(1001, 'Python'),
(1001, 'Power BI'),
(1002, 'Java'),
(1002, 'SQL'),
(1003, 'Python');

++
||
++
++

In [35]:
%%sql

SELECT *
FROM test.student1 s1
JOIN test.student1_subject s1s
    ON s1.student_id = s1s.student_id;

student_id,student_name,student_id_1,subject_name
1001,千尋,1001,DB
1001,千尋,1001,Power BI
1001,千尋,1001,Python
1002,ハウル,1002,Java
1002,ハウル,1002,SQL
1003,キキ,1003,Python


In [36]:
%%sql

-- 列を追加せずに、新しい科目を追加できる
INSERT INTO test.student1_subject
VALUES (1001, 'Java');

++
||
++
++

In [37]:
%%sql

SELECT *
FROM test.student1 s1
JOIN test.student1_subject s1s
    ON s1.student_id = s1s.student_id;

student_id,student_name,student_id_1,subject_name
1001,千尋,1001,DB
1001,千尋,1001,Java
1001,千尋,1001,Power BI
1001,千尋,1001,Python
1002,ハウル,1002,Java
1002,ハウル,1002,SQL
1003,キキ,1003,Python
